# Task 3: Running a model locally with Ollama, fp16 vs 4-bit (q4_K_M)
Runs on a free Colab T4 (Runtime -> Change runtime type -> T4 GPU). The same code works on your own laptop:
skip the install cell, make sure `ollama serve` is running, and run the rest.

In [1]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
import subprocess, time, requests, json, pandas as pd
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
URL = "http://localhost:11434"
MODELS = {"fp16": "qwen2.5:1.5b-instruct-fp16", "q4_K_M": "qwen2.5:1.5b-instruct-q4_K_M"}
for m in MODELS.values(): subprocess.run(["ollama", "pull", m], check=True)

## 10 prompts (same for both settings): mix of facts, reasoning, code, format-following, summarising

In [3]:
PROMPTS = [
 "What is the capital of Australia, and why is it not Sydney?",
 "Explain what a hash map is to a 12-year-old in 4 sentences.",
 "A train leaves at 14:20 and arrives at 17:05. How long is the journey? Show your working.",
 "Write a Python function that returns the n-th Fibonacci number iteratively, with a docstring.",
 "Extract name, city and age as JSON only: 'Ananya, 29, moved from Pune to Jabalpur last year.'",
 "Translate to French: 'Could you please send me the report by Friday afternoon?'",
 "Summarise in one sentence: 'Quantisation reduces the numerical precision of a neural network's weights, "
 "for example from 16-bit floats to 4-bit integers, shrinking memory use and often speeding up inference at a small cost in accuracy.'",
 "Write a four-line poem about monsoon rain without using the letter 'z'.",
 "If all bloops are razzies and all razzies are lazzies, are all bloops lazzies? Explain briefly.",
 "Give me 3 pros and 3 cons of remote work as a bulleted list.",
]

## Benchmark: tokens/sec (from Ollama's own timers) and memory (Ollama `/api/ps` + `nvidia-smi`)

In [4]:
def gen(model, prompt, num_predict=200):
    return requests.post(f"{URL}/api/generate", json={
        "model": model, "prompt": prompt, "stream": False, "keep_alive": "10m",
        "options": {"temperature": 0, "seed": 0, "num_predict": num_predict}}).json()

def gpu_mem_mb():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.strip()
        return int(out.split("\n")[0]) if out else None
    except Exception:
        return None

def unload(model):
    requests.post(f"{URL}/api/generate", json={"model": model, "keep_alive": 0}); time.sleep(3)

records, outputs, mem = [], {}, {}
for label, model in MODELS.items():
    unload_base = gpu_mem_mb()
    gen(model, "Hello")                                  # warm-up (loads the model)
    ps = [m for m in requests.get(f"{URL}/api/ps").json()["models"] if m["name"].startswith(model.split(":")[0])][0]
    mem[label] = {"ollama_total_GB": ps["size"] / 1e9, "ollama_vram_GB": ps.get("size_vram", 0) / 1e9,
                  "nvidia_smi_used_MB": gpu_mem_mb() or 0}
    outputs[label] = []
    for p in PROMPTS:
        r = gen(model, p)
        outputs[label].append(r["response"].strip())
        records.append({"setting": label, "tokens": r["eval_count"],
                        "tok_per_s": r["eval_count"] / r["eval_duration"] * 1e9})
    unload(model)

df = pd.DataFrame(records)
speed = df.groupby("setting").tok_per_s.agg(["mean", "min", "max"]).round(1)
print(speed); print(pd.DataFrame(mem).T.round(2))

          mean    min    max
setting                     
fp16      60.7   59.0   62.1
q4_K_M   128.0  121.6  132.6
        ollama_total_GB  ollama_vram_GB  nvidia_smi_used_MB
fp16               3.28            3.28              3263.0
q4_K_M             1.17            1.17              1243.0


## Quality comparison: read both outputs for each prompt and score 1-5 yourself
Judge: correctness, following the format/constraint, fluency. Look for where 4-bit differs (arithmetic, strict constraints, code).

In [5]:
for i, p in enumerate(PROMPTS):
    print(f"\n===== Prompt {i+1}: {p}")
    for label in MODELS: print(f"\n--- {label} ---\n{outputs[label][i]}")


===== Prompt 1: What is the capital of Australia, and why is it not Sydney?

--- fp16 ---
The capital of Australia is Canberra. Sydney is the most populous city in Australia, but it is not the capital. Canberra was established in 1908 as the capital of the Australian Commonwealth, and it is located in the Australian Capital Territory.

--- q4_K_M ---
The capital of Australia is Canberra. Sydney is the most populous city in Australia, but it is not the capital. Canberra is the seat of the Australian Government and is located in the Australian Capital Territory.

===== Prompt 2: Explain what a hash map is to a 12-year-old in 4 sentences.

--- fp16 ---
A hash map is like a special kind of dictionary where you can quickly find the word you're looking for. Imagine you have a big box of toys, and each toy has a number on it. If you want to find a specific toy, you can look at the number and quickly find it in the box. In a hash map, instead of looking in a box, you look at a number (called 

In [6]:
# Fill these in after reading the outputs above (1 = bad, 5 = perfect), one number per prompt:
SCORES = {"fp16":   [0]*10,    # TODO
          "q4_K_M": [0]*10}    # TODO
print({k: sum(v)/len(v) for k, v in SCORES.items()})

{'fp16': 0.0, 'q4_K_M': 0.0}


In [7]:
pd.DataFrame({"prompt": PROMPTS, "fp16_output": outputs["fp16"], "q4_output": outputs["q4_K_M"],
              "fp16_score": SCORES["fp16"], "q4_score": SCORES["q4_K_M"]}).to_csv("task3_quality.csv", index=False)
speed.to_csv("task3_speed.csv"); pd.DataFrame(mem).T.to_csv("task3_memory.csv")
from google.colab import files
for f in ["task3_quality.csv", "task3_speed.csv", "task3_memory.csv"]: files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>